# Continuing Syncfusion .NET CLI Development

A code-focused notebook for exploring and extending the Syncfusion .NET CLI while respecting its command structure, platform boundaries, and validation requirements.

> Keep investigation limited to the requested feature. Before implementation, inspect the current working tree and relevant code so existing edits are not overwritten or broadened.

## 1. Inspect the .NET CLI Solution Structure

Start by checking the repository state and solution/project files. Inspect only files tied to the requested behavior; note pre-existing modifications before editing.

The expected boundaries are:

- CLI driver and `System.CommandLine` handlers: `src/Syncfusion.DotNet.Cli`
- Platform-neutral contracts, requests, paths, and process helpers: `src/Syncfusion.DotNet.Cli.Core`
- MAUI project discovery, catalog, package and template behavior: `src/Platforms/Syncfusion.DotNet.Cli.Maui`
- Focused regression and integration coverage: `tests/Syncfusion.DotNet.Cli.Tests`

Use a terminal cell to inspect the solution and targeted files. Avoid broad file dumps when a scoped search answers the question.

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path.cwd()
print(f"Workspace: {ROOT}")
print(subprocess.run(["git", "status", "--short"], cwd=ROOT, text=True, capture_output=True, check=True).stdout or "Working tree clean")
print("\nSolution/project files:")
for path in sorted([*ROOT.glob("*.slnx"), *ROOT.glob("src/**/*.csproj"), *ROOT.glob("tests/**/*.csproj")]):
    print(path.relative_to(ROOT))

## 2. Trace a Command from Parsing to Execution

Choose a command related to the feature, then trace its flow:

1. Find the `System.CommandLine` definition, arguments, options, validation, and handler.
2. Identify the Core request or contract passed by the handler.
3. Follow the MAUI operation/service that performs the work.
4. Follow the presenter/result path and check that `--json` remains clean machine-readable output.
5. Compare the direct command with the interactive menu path. Prefer one shared operation path; preserve non-interactive behavior and compatibility forms.

A focused source search is a practical starting point; substitute the command or symbol being investigated.

In [ ]:
# Set a narrow search term after selecting the command or operation to trace.
SEARCH_TERM = "ControlInstaller"  # Replace with the target command, request, or service symbol.
result = subprocess.run(
    ["rg", "-n", SEARCH_TERM, "src/Syncfusion.DotNet.Cli", "src/Syncfusion.DotNet.Cli.Core", "src/Platforms/Syncfusion.DotNet.Cli.Maui"],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout if result.returncode == 0 else "No matches; refine SEARCH_TERM or source paths.")

## 3. Define a Core Request and Command Contract

For new behavior, extend the appropriate contract in Core before wiring implementation. Keep the request platform-neutral: no MAUI types, project-specific paths, or platform process logic in shared Core. Preserve the CLI's driver → command → arguments/options organization and existing command compatibility.

Before changing a contract, inspect its callers and tests. Prefer reusing an existing request or service when it already expresses the behavior; avoid parallel direct and interactive implementations.

## 4. Implement MAUI-Specific Behavior

Implement MAUI-specific work in `src/Platforms/Syncfusion.DotNet.Cli.Maui`, reusing existing discovery, execution, package, template, manifest, and rollback services. Keep package-only actions distinct from template actions.

Use .NET path and process APIs (`Path`, `DirectoryInfo`, existing process runners). Do not hardcode operating-system paths, shell syntax, separators, or executable suffixes. For operations that modify project state, plan the mutation boundary and rollback behavior before writing files. Treat external command output and project XML as untrusted input; validate expected structure and values.

## 5. Connect Direct and Interactive Workflows

Wire argument parsing and validation into the shared Core/MAUI operation path already used by the corresponding interactive action. Preserve existing command forms unless an explicit compatibility change is part of the request. Keep prompts out of JSON output, redirected/noninteractive execution, and other modes that explicitly disable interaction.

Validate the user-facing result through the existing presenter rather than writing ad hoc console output from business logic. Keep stdout free of diagnostics when JSON output is requested.

## 6. Add Focused Tests

Add regressions that prove the requested behavior and its safety boundaries: valid operation, input/configuration validation, output contract, failure handling, and rollback as applicable. Prefer existing fake runners and test seams; keep tests scoped to the change. Run the affected tests after each meaningful increment. Never disable tests, analyzers, nullable checking, or warnings-as-errors to make a change pass.

## 7. Validate Formatting, Build, Tests, and Package

At category completion, validate in the repository's required order and report actual outcomes:

1. Format verification (do not apply broad formatting outside the change).
2. Release build.
3. Affected tests, then the full test suite once.
4. Package once.
5. `git diff --check` and final diff/status review.

Review each result for formatting issues, build errors, compiler/analyzer/nullable warnings, and failed tests. Follow `global.json` and solution instructions. If any step fails, investigate and fix the root cause, then rerun the relevant affected checks before proceeding. Do not claim completion without evidence.